In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("sets.csv", sep=',')
print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())

(14936, 17)
  Set_ID            Name  Year      Theme Theme_Group   Subtheme Category  \
0   75-1   PreSchool Set  1975  PreSchool  Pre-school        NaN   Normal   
1   77-1   PreSchool Set  1975  PreSchool  Pre-school        NaN   Normal   
2  077-1  Pre-School Set  1975      Duplo  Pre-school        NaN   Normal   
3   78-1   PreSchool Set  1975  PreSchool  Pre-school        NaN   Normal   
4   78-3       Basic Set  1975  Samsonite     Vintage  Basic set   Normal   

         Packaging  Num_Instructions     Availability  Pieces  Minifigures  \
0  {Not specified}                 0  {Not specified}    16.0          NaN   
1  {Not specified}                 0  {Not specified}    20.0          NaN   
2  {Not specified}                 0  {Not specified}    21.0          NaN   
3  {Not specified}                 0  {Not specified}    32.0          NaN   
4              Box                 0  {Not specified}   330.0          NaN   

   Owned  Rating  USD_MSRP  Total_Quantity  Current_Pric

As can be seen above, our dataset consists of 17 columns: 7 floats, 8 strings, and 2 integers and 14936 rows.  
Then, there are numerous columns that contain null values, notably USD-MSRP and Current_price.  
We will therefore need to remove a large portion of the dataset during the cleaning phase; since our goal is to predict the price, it would be too risky to keep these columns or to artificially generate the prices.


In [5]:
(df.isna().sum() == 0).sum()

np.int64(9)

In [6]:
(df["USD_MSRP"].isnull()|df["Year"].isnull()|df["Theme"].isnull()|df["Subtheme"].isnull()|df["Theme_Group"].isnull()|df["Category"].isnull()|df["Packaging"].isnull()|df["Num_Instructions"].isnull()|df["Owned"].isnull()|df["Rating"].isnull()).sum()

np.int64(10005)

We discovered that there are only 9 rows containing all the information, so we need to select columns. The first arbitrary selection we made is as follows: [USD_MSRP, Year, Theme, Subtheme, Theme_group, Category, Packaging, Num_Instructions, Pieces, Owned, Rating]. With this selection of columns, there are 10,005 incomplete columns, so we end up with a dataset containing 14,936 – 10,005 = 4,931 clean rows.

# Clean the data

In [7]:
print(f"Number of duplicates : {df.duplicated().sum()}")

Number of duplicates : 0


In [8]:
colonnes = [
    "USD_MSRP", "Year", "Theme", "Subtheme", "Theme_Group",
    "Category", "Packaging", "Num_Instructions", "Owned", "Rating"
]

df_clean = df[colonnes].dropna().copy()

print(df_clean.shape)

(4931, 10)


In [9]:
df_clean[["Num_Instructions","Year"]] = df_clean[["Num_Instructions","Year"]].astype(float)

In [10]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 4931 entries, 1706 to 14935
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   USD_MSRP          4931 non-null   float64
 1   Year              4931 non-null   float64
 2   Theme             4931 non-null   str    
 3   Subtheme          4931 non-null   str    
 4   Theme_Group       4931 non-null   str    
 5   Category          4931 non-null   str    
 6   Packaging         4931 non-null   str    
 7   Num_Instructions  4931 non-null   float64
 8   Owned             4931 non-null   float64
 9   Rating            4931 non-null   float64
dtypes: float64(5), str(5)
memory usage: 423.8 KB


In [11]:
df_clean.head(10)

,USD_MSRP,Year,Theme,Subtheme,Theme_Group,Category,Packaging,Num_Instructions,Owned,Rating
1706,12.99,1991.0,Trains,9V,Modern day,Normal,Box,0.0,3024.0,4.0
1707,12.99,1991.0,Trains,9V,Modern day,Normal,Box,0.0,2934.0,3.8
1708,27.99,1991.0,Trains,9V,Modern day,Normal,Box,0.0,3047.0,4.0
1713,41.99,1991.0,Trains,9V,Modern day,Normal,Box,0.0,2162.0,3.7
1957,4.99,1993.0,Duplo,Trains,Pre-school,Normal,{Not specified},0.0,470.0,0.0
1958,4.99,1993.0,Duplo,Trains,Pre-school,Normal,{Not specified},0.0,449.0,0.0
2397,4.99,1996.0,Basic,Supplementaries,Basic,Normal,Other,0.0,3636.0,4.1
2399,14.99,1996.0,Basic,Supplementaries,Basic,Normal,Shrink-wrapped,0.0,3964.0,4.0
2401,2.49,1996.0,Basic,Supplementaries,Basic,Normal,Polybag,0.0,2137.0,4.4
2787,6.99,1997.0,Town,Accessories,Modern day,Normal,Shrink-wrapped,0.0,754.0,3.9


We need to check whether there are any categories in the category column that indicate we don't have any information—such as the {Not specified} category under “Packaging.”

In [13]:
df_clean["Theme"].unique()

<StringArray>
[                      'Trains',                        'Duplo',
                        'Basic',                         'Town',
                        'Dacta',                  'Bulk Bricks',
                         'Gear',                    'Star Wars',
                     'Seasonal',                         'City',
                       'Racers',                     'Bionicle',
                       'Castle',                      'Creator',
                    'Exo-Force',                   'Mindstorms',
              'Advanced models',                 'Harry Potter',
                     'Belville',                        'Space',
                    'Education',                'Indiana Jones',
              'Power Functions',                      'Factory',
                 'Architecture',                'Miscellaneous',
                        'Games',                      'Pirates',
                        'Books',                 'HERO Factory',
           

In [16]:
df_clean["Subtheme"].unique()

<StringArray>
[                               '9V',                            'Trains',
                   'Supplementaries',                       'Accessories',
                            'System',                          'Seasonal',
                            'Castle',              'Key Chains/Star Wars',
                             'Space',         'Ultimate Collector Series',
 ...
                  'The Way of Water',                             'Mechs',
    'Guardians of the Galaxy Vol. 3', 'Ant-Man and the Wasp: Quantumania',
                     'House Banners',                            'Pagani',
                            'Nissan',            'Key Chains/Minifigures',
                 'Key Chains/Marvel',              'Key Chains/Minecraft']
Length: 551, dtype: str

In [12]:
df_clean["Theme_Group"].unique()

<StringArray>
[      'Modern day',       'Pre-school',            'Basic',
      'Educational',    'Miscellaneous',         'Licensed',
           'Racing',     'Constraction',       'Historical',
     'Model making', 'Action/Adventure',        'Technical',
            'Girls',           'Junior',   'Art and crafts']
Length: 15, dtype: str

In [22]:
df_clean["Category"].unique()

<StringArray>
['Normal', 'Gear', 'Other', 'Collection', 'Extended', 'Book', 'Random']
Length: 7, dtype: str

In [23]:
df_clean["Packaging"].unique()

<StringArray>
[                  'Box',       '{Not specified}',                 'Other',
        'Shrink-wrapped',               'Polybag',      'Plastic canister',
           'Plastic box',              'Canister',                   'Tag',
          'Blister pack',             'Foil pack',                'Bucket',
                   'Tub', 'Box with backing card',       'Box with handle']
Length: 15, dtype: str

Upon closer inspection, we can see that the only column causing a problem is {Not specified} under “Packaging,” but we've decided to keep it because the lack of information is, after all, a form of information. It is, after all, an adjustment variable that we need to keep in mind.

## Outliers

In [28]:
def detect_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1

    mask = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    outliers = df_clean.loc[s.index[mask]]

    print(f"Number of outliers : {len(outliers)}")
    print(outliers[[s.name]].head())


numeric_cols = [
    "USD_MSRP",
    "Year",
    "Num_Instructions",
    "Owned",
    "Rating",
]

for col in numeric_cols:
    if col in df_clean.columns:
        print(f"Outliers for {col}:")
        detect_outliers(df_clean[col])
        print("\n")

Outliers for USD_MSRP:
Number of outliers : 449
      USD_MSRP
4769    269.99
5324    249.99
5973    269.99
5974    139.99
6167     99.99


Outliers for Year:
Number of outliers : 9
        Year
1706  1991.0
1707  1991.0
1708  1991.0
1713  1991.0
1957  1993.0


Outliers for Num_Instructions:
Number of outliers : 140
      Num_Instructions
5828              10.0
7945               9.0
8060              12.0
8129              10.0
8206              12.0


Outliers for Owned:
Number of outliers : 269
        Owned
5826  13782.0
5830  13471.0
6161  12210.0
6168  11955.0
6654  13070.0


Outliers for Rating:
Number of outliers : 0
Empty DataFrame
Columns: [Rating]
Index: []




We observe that only the “Rating” column has no outliers when using the IQR method; however, the other columns have outliers that may be from the year 1991, and it is possible that sets priced at $269 actually exist. The same is true for ‘Owned’ and “Num_instruction,” which have no outliers.

To clean up the dataset, there isn't much we need to do because there are no duplicate rows, and after selecting the relevant columns and removing rows with missing values, there are no anomalies or floating-point numbers that aren't recognized. 

# Preprocessing

Now we need to separate the data into X and Y. We'll use USD_MSRP as our target value.

In [32]:
X = df_clean.drop(columns=["USD_MSRP"])
y = df_clean["USD_MSRP"]

We decided to split our dataset into three parts so that we could adjust the parameters using validation data and perform final testing using the test set.

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

We use OneHotEncoder to convert each categorical variable without a natural order into several binary columns (0 or 1). This prevents a linear regression model from interpreting numerical codes (0, 1, 2...) as an order or an increasing scale between categories.

In [34]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore')

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

## Pipeline

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = X[
    'Year',
    'Num_Instructions',
    'Owned',
    'Rating'
]

categorical_features = X[
    'Theme',
    'Subtheme',
    'Theme_Group',
    'Category',
    'Packaging'
]

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

KeyError: ('Year', 'Num_Instructions', 'Owned', 'Rating')